In [ ]:
!pip install -q langchain pydantic python-dotenv openai tiktoken faiss-cpu chromadb lark pymongo sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 56.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.8/479.8 kB 30.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.3/671.3 kB 43.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 7.0 MB/s eta

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/_openai/
import os
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.
%cd /content/

Mounted at /content/drive
/content/drive/MyDrive/_openai
/content


In [ ]:
import datetime
import pymongo
from pymongo import MongoClient

uri = os.environ['MONGODB_URI']

client = MongoClient( uri )

In [ ]:
# client.stats

In [ ]:
# List all database names
client.list_database_names()

['blnimythuypydrs']

In [ ]:
db = client["blnimythuypydrs"]
db.list_collection_names()

['CVs']

In [ ]:
import openai
# openai.api_key

from pydantic import BaseModel, Field

from langchain.chat_models import ChatOpenAI
from langchain.agents import Tool
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

from transformers import AutoTokenizer, AutoModel

from sklearn.metrics.pairwise import cosine_similarity

from pprint import pprint
import json

In [ ]:
#@title Embedding with HF model
embedding_model="thenlper/gte-base"
hf_tokenizer = AutoTokenizer.from_pretrained(embedding_model)
hf_model = AutoModel.from_pretrained(embedding_model)

def embedding_text(input_text):
    batch_dict = hf_tokenizer([input_text.strip().lower()], max_length=512, padding=True, truncation=True, return_tensors='pt')
    outputs = hf_model(**batch_dict)
    last_hidden = outputs.last_hidden_state.masked_fill(~batch_dict['attention_mask'][..., None].bool(), 0.0)
    torch_embeddings_list = last_hidden.sum(dim=1) / batch_dict['attention_mask'].sum(dim=1)[..., None]
    return torch_embeddings_list[0].tolist() # dimension of 768

In [ ]:
#@title Embedding with OpenAI model
embedding_model="text-embedding-ada-002"

def embedding_text(input_text):
    response = openai.Embedding.create(input=input_text,model=embedding_model)
    embeddings = response['data'][0]['embedding']
    return embeddings

# Creating a new collection

In [ ]:
# Create the schema for the collection
collection_name = "CVs"
db[collection_name].drop()
collection = db[collection_name]

In [ ]:
list(collection.find({}))

[]

In [ ]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings()

# Change the current directory to the specified path
%cd /content/drive/MyDrive/_openai/
directory_path = "/content/drive/MyDrive/_openai/"

# Create a dictionary to store Faiss indices
vector_stores = {}

# Iterate through files in the directory
for dir_name in os.listdir(directory_path):
    # Check if the directory name ends with "_faiss_index"
    if dir_name.endswith("_faiss_index"):
        # Construct the full index path
        index_path = os.path.join(directory_path, dir_name)

        # Extract the key from the directory name and use it as a dictionary key
        vector_stores[dir_name.split('_')[0]] = FAISS.load_local(index_path, embeddings)

# Change the current directory back to the original directory
%cd /content/

# Create an empty list to store documents
list_docs = []

# Search for similar documents across all Faiss indices
# Extend the list_docs with the results of similarity searches
[ list_docs.extend(v.similarity_search(" ", k=99)) for v in vector_stores.values() ]

# Get the total number of documents in the list
len(list_docs)

/content/drive/MyDrive/_openai
/content


10

In [ ]:
#@title Let's ingest the data

def ingest_cv_documents(list_of_documents, mongo_collection):
    # A list with entities
    list_of_entities = []
    for document in list_of_documents:
        entity = {}
        embeddings = embedding_text(str(document.metadata)) # A numerical vector
        entity['cv_full_text'] = document.page_content # cv's full text
        entity['cv_metadata'] = document.metadata # cv's metadata JSON format
        entity['cv_metadata_str'] = str(document.metadata) # cv's metadata
        entity['embedding_metadata'] = embeddings # embeddings of cvs's metadata
        list_of_entities.append(entity)

    result = mongo_collection.insert_many(list_of_entities)
    if result.acknowledged:
        print(f"{result.inserted_ids} entities were inserted")
    else:
        print("No entity was inserted.")

ingest_cv_documents(list_docs, collection)

[ObjectId('653919fbda6c432e291f1b5d'), ObjectId('653919fbda6c432e291f1b5e'), ObjectId('653919fbda6c432e291f1b5f'), ObjectId('653919fbda6c432e291f1b60'), ObjectId('653919fbda6c432e291f1b61'), ObjectId('653919fbda6c432e291f1b62'), ObjectId('653919fbda6c432e291f1b63'), ObjectId('653919fbda6c432e291f1b64'), ObjectId('653919fbda6c432e291f1b65'), ObjectId('653919fbda6c432e291f1b66')] entities were inserted


# Loading existing collection

In [ ]:
# Create the schema for the collection
collection_name = "CVs"
collection = db[collection_name]
len(list(collection.find({})))

10

# Searching similar

In [ ]:
def find_similar(user_intent, k=3):
    embedding_user_intent = embedding_text(user_intent)
    # Query for similar documents based on the "embedding_metadata" field
    query = {
        "embedding_metadata": {
            "$near": {
                "$geometry": {
                    "type": "Point",
                    "coordinates": embedding_user_intent
                },
                "$maxDistance": 1  # Adjust this value to control similarity threshold
            }
        }
    }

    # Sort the results by similarity (nearest first) and limit the number of results to k
    cursor = collection.find(query).sort([("score", pymongo.DESCENDING)]).limit(k)
    print(type(cursor))
    # Retrieve and return the similar documents
    similar_documents = list(cursor)
    return similar_documents

In [ ]:
find_similar("'proffesion_field': 'Graphic Designer', 'country': 'Peru', 'genre': 'Female'", k = 1)

<class 'pymongo.cursor.Cursor'>


OperationFailure: ignored

In [ ]:

def find_similar_2(user_intent, k=3):
    embedding_user_intent = embedding_text(user_intent)  # Assuming this function returns the embedding vector

    # Fetch all documents from the collection
    all_documents = list(collection.find({}))

    # Calculate cosine similarity between user's embedding and all documents' embeddings
    similarities = []
    for doc in all_documents:
        doc_embedding = doc["embedding_metadata"]  # Replace with the actual field name
        similarity = cosine_similarity([embedding_user_intent], [doc_embedding])
        similarities.append((doc, similarity))

    # Sort the results by similarity and limit the number of results to k
    sorted_results = sorted(similarities, key=lambda x: x[1], reverse=True)[:k]

    # Retrieve and return the similar documents
    similar_documents = [result[0] for result in sorted_results]
    return similar_documents

# Example usage:
similar_docs = find_similar_2("'proffesion_field': 'Graphic Designer', 'country': 'Peru', 'genre': 'Female'", k=1)
for doc in similar_docs:
    print(doc['cv_metadata'])
print('\n')
similar_docs = find_similar_2("'country': 'Peru', 'genre': 'Male'", k=3)
for doc in similar_docs:
    print(doc['cv_metadata'])

{'source': '/content/cv-ejemplo-disenador-grafico.pdf', 'page': 0, 'full_name': 'Raquel Vázquez García', 'proffesion_field': 'Graphic Designer', 'country': 'Peru', 'genre': 'Female', 'languages': ['Spanish', 'English'], 'years_experience': '2', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': [], 'hard_skills': ['CorelDRAW', 'Adobe Illustrator', 'Affinity Publisher']}


{'source': '/content/cv-ejemplo-profesor-2a9d8f.pdf', 'page': 0, 'full_name': 'Jorge Gaya', 'proffesion_field': 'Private Tutor', 'country': 'Peru', 'genre': 'Male', 'languages': ['Spanish'], 'years_experience': '10', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Communication'], 'hard_skills': ['Planning and Methodology', 'Pedagogical Knowledge', 'Topic Preparation']}
{'source': '/content/cv-ejemplo-abogado-406692.pdf', 'page': 0, 'full_name': 'Patricio Ugarte', 'proffesion_field': 'Law', 'country': 'Peru', 'genre': 'Male', 'languages': ['Spanish'], 'years_experience': 14, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['

# search for elements in a collection

In [ ]:
list(collection.find({},{"cv_metadata":1,"_id":0}).limit(1))

[{'cv_metadata': {'source': '/content/cv-ejemplo-abogado-406692.pdf',
   'page': 0,
   'full_name': 'Patricio Ugarte',
   'proffesion_field': 'Law',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish'],
   'years_experience': 14,
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Clear communication', 'Idealism', 'Morality'],
   'hard_skills': ['Labor law', 'Legal research', 'Litigation']}}]

In [ ]:
list(collection.find({},{"cv_metadata_str":1,"_id":0}).limit(1))

[{'cv_metadata_str': "{'source': '/content/cv-ejemplo-abogado-406692.pdf', 'page': 0, 'full_name': 'Patricio Ugarte', 'proffesion_field': 'Law', 'country': 'Peru', 'genre': 'Male', 'languages': ['Spanish'], 'years_experience': 14, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Clear communication', 'Idealism', 'Morality'], 'hard_skills': ['Labor law', 'Legal research', 'Litigation']}"}]

In [ ]:
list(collection.find({"cv_metadata.country":"Peru"},{"_id":1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5d')},
 {'_id': ObjectId('653919fbda6c432e291f1b5e')},
 {'_id': ObjectId('653919fbda6c432e291f1b5f')},
 {'_id': ObjectId('653919fbda6c432e291f1b60')},
 {'_id': ObjectId('653919fbda6c432e291f1b64')},
 {'_id': ObjectId('653919fbda6c432e291f1b65')}]

In [ ]:
list(collection.find({"cv_metadata.country":"peru"},{"_id":1}))

[]

In [ ]:
list(collection.find({"cv_metadata.country": {"$regex": "peru", "$options": "i"}}, {"_id": 1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5d')},
 {'_id': ObjectId('653919fbda6c432e291f1b5e')},
 {'_id': ObjectId('653919fbda6c432e291f1b5f')},
 {'_id': ObjectId('653919fbda6c432e291f1b60')},
 {'_id': ObjectId('653919fbda6c432e291f1b64')},
 {'_id': ObjectId('653919fbda6c432e291f1b65')}]

In [ ]:
list(collection.find({"cv_metadata.full_name":"Patricio Ugarte"},        {"_id":1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5d')}]

In [ ]:
list(collection.find({"cv_metadata.full_name": {"$regex": "Patricio", "$options": "i"}}, {"_id": 1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5d')}]

In [ ]:
list(collection.find({"cv_metadata.full_name": {"$regex": "ugarte", "$options": "i"}}, {"_id": 1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5d')}]

In [ ]:
list(collection.find({"cv_metadata.full_name": {"$regex": "ugarte", "$options": "i"}} ,{'cv_metadata':1}).limit(1))

[{'_id': ObjectId('653919fbda6c432e291f1b5d'),
  'cv_metadata': {'source': '/content/cv-ejemplo-abogado-406692.pdf',
   'page': 0,
   'full_name': 'Patricio Ugarte',
   'proffesion_field': 'Law',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish'],
   'years_experience': 14,
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Clear communication', 'Idealism', 'Morality'],
   'hard_skills': ['Labor law', 'Legal research', 'Litigation']}}]

In [ ]:
list(collection.find({"cv_metadata.languages": {"$regex": "english", "$options": "i"}}, {"_id": 1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5e')},
 {'_id': ObjectId('653919fbda6c432e291f1b5f')},
 {'_id': ObjectId('653919fbda6c432e291f1b60')},
 {'_id': ObjectId('653919fbda6c432e291f1b61')},
 {'_id': ObjectId('653919fbda6c432e291f1b63')},
 {'_id': ObjectId('653919fbda6c432e291f1b65')},
 {'_id': ObjectId('653919fbda6c432e291f1b66')}]

In [ ]:
list(collection.find({"cv_metadata.hard_skills": {"$regex":"law|planning", "$options": "i"}}, {'cv_metadata':1, "_id": 1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5d'),
  'cv_metadata': {'source': '/content/cv-ejemplo-abogado-406692.pdf',
   'page': 0,
   'full_name': 'Patricio Ugarte',
   'proffesion_field': 'Law',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish'],
   'years_experience': 14,
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Clear communication', 'Idealism', 'Morality'],
   'hard_skills': ['Labor law', 'Legal research', 'Litigation']}},
 {'_id': ObjectId('653919fbda6c432e291f1b64'),
  'cv_metadata': {'source': '/content/cv-ejemplo-profesor-2a9d8f.pdf',
   'page': 0,
   'full_name': 'Jorge Gaya',
   'proffesion_field': 'Private Tutor',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish'],
   'years_experience': '10',
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Communication'],
   'hard_skills': ['Planning and Methodology',
    'Pedagogical Knowledge',
    'Topic Preparation']}}]

In [ ]:
# gt, lt
list(collection.find({"cv_metadata.years_experience": {"$gt": 10}}, {'cv_metadata':1, "_id": 1}))

[{'_id': ObjectId('653919fbda6c432e291f1b5d'),
  'cv_metadata': {'source': '/content/cv-ejemplo-abogado-406692.pdf',
   'page': 0,
   'full_name': 'Patricio Ugarte',
   'proffesion_field': 'Law',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish'],
   'years_experience': 14,
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Clear communication', 'Idealism', 'Morality'],
   'hard_skills': ['Labor law', 'Legal research', 'Litigation']}},
 {'_id': ObjectId('653919fbda6c432e291f1b5e'),
  'cv_metadata': {'source': '/content/cv-ejemplo-arquitecto-333333.pdf',
   'page': 0,
   'full_name': 'Diego García',
   'proffesion_field': 'Architect',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish', 'English'],
   'years_experience': 18,
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Quick comprehension',
    'Anticipating needs',
    'Passion for the profession'],
   'hard_skills': ['AutoCAD', 'Revit', 'Grasshopper', 'Adobe Photoshop']}}]

In [ ]:
import json

input_str = '{"cv_metadata.years_experience": {"$gt": 10, "$lt": 15}, "cv_metadata.hard_skills": {"$regex":"regulations|law", "$options": "i"} }'

# Replace single quotes with double quotes to make it valid JSON
json_str = input_str.replace("'", '"')

# Parse the JSON string into a Python dictionary
data_dict = json.loads(json_str)

# Now, 'data_dict' contains the parsed dictionary
print(type(data_dict))


<class 'dict'>


In [ ]:
query = {"cv_metadata.years_experience": {"$gt": 10, "$lt": 15}, "cv_metadata.hard_skills": {"$regex":"regulations|law", "$options": "i"} }

# 'cv_full_text' # cv's full text
# 'cv_metadata' # cv's metadata JSON format
# 'cv_metadata_str' # cv's metadata
# 'embedding_metadata' # embeddings of cvs's metadata
def search_candidates_by(user_query, limit=99, retrieve_data='cv_metadata'):
    if isinstance(user_query, str):  # Changed "is" to "isinstance" and added "str" to the condition.
        user_query = json.loads(user_query.replace("'",'"'))  # Fixed the JSON loading.
    list_collections = list(collection.find(user_query, {retrieve_data:1, "_id": 0}).limit(limit))
    return list_collections

available_functions = {
    'search_candidates_by': search_candidates_by
}

search_candidates_by(query)

[{'cv_metadata': {'source': '/content/cv-ejemplo-abogado-406692.pdf',
   'page': 0,
   'full_name': 'Patricio Ugarte',
   'proffesion_field': 'Law',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish'],
   'years_experience': 14,
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Clear communication', 'Idealism', 'Morality'],
   'hard_skills': ['Labor law', 'Legal research', 'Litigation']}}]

In [ ]:
search_candidates_by('{"cv_metadata.years_experience": {"$gt": 10, "$lt": 15}, "cv_metadata.hard_skills": {"$regex":"regulations|law", "$options": "i"} }')

[{'cv_metadata': {'source': '/content/cv-ejemplo-abogado-406692.pdf',
   'page': 0,
   'full_name': 'Patricio Ugarte',
   'proffesion_field': 'Law',
   'country': 'Peru',
   'genre': 'Male',
   'languages': ['Spanish'],
   'years_experience': 14,
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Clear communication', 'Idealism', 'Morality'],
   'hard_skills': ['Labor law', 'Legal research', 'Litigation']}}]

In [ ]:
result = search_candidates_by(query,retrieve_data='cv_full_text')
result[0]['cv_full_text']

'Patricio Ugarte\nDetalles personales\nNombre Patricio Ugarte\nDirección Calle bajo 10, Lima 111321\nNúmero de teléfono +51 0123 456 789\nCorreo electrónico ejemplo@cvmaker.pe\nPerfil\nAbogado idealista apasionado por la profesión legal. Con una sólida formación en todos los aspectos del derecho laboral, recibo\nnuevos clientes sin barreras. Me comunico con un lenguaje sencillo porque creo que la ley debe ser accesible y comprensible para\ntodos. ¡Comunicación clara con un firme apretón de manos!\nEducación\nMaestría en Derecho Laboral may. 2012 - nov. 2021\nUniversidad de Lima, Lima\nExperiencia laboral\nAbogado may. 2012 - nov. 2021\nGonzález Abogados, Lima\nComo especialista en derecho laboral, ha adquirido una amplia experiencia tanto en la práctica consultiva\ncomo en la litigación. Con un buen sentido del idealismo y la moralidad, se resolvieron innumerables conflictos\nentre empleados y patrones. El trabajo abarcó todos los aspectos del derecho laboral, incluida la ley de\ndespi

In [ ]:
functions = [
        {
            "name": "search_candidates_by",
            "description": "Get a list of candidates which match better with the user's intent",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_query": {
                        "type": "string",
                        "description": """"
A JSON string query for mongodb and always use regex and ignore case as options for each string field,
You must always generate it, applying ignore case and regex for every field, use "$regex", "$options"
The collection structure has a field 'cv_metadata' dictionary which contains:

  - full_name: string
  - proffesion_field: string
  - country: string
  - genre: string
  - languages: list(string)
  - years_experience: integer
  - email: string
  - soft_skills: list(string)
  - hard_skills: list(string)

  for example:
    - {"cv_metadata.years_experience": {"$gt": 10, "$lt": 15}, "cv_metadata.hard_skills": {"$regex":"regulations|law", "$options": "i"} }
    - {"cv_metadata.field": {"$regex":"value", "$options": "i"}}
                        """,
                    },
                    "limit": {
                        "type": "integer",
                        "description": """
The number of candidates the user is looking for is set to 3 by default.
If the user requests multiple candidates.
If the user only requests one candidate or uses the singular form, it is set to 1 by default.
If the user explicitly mentions the number of candidates, that value is returned
"""
                    }
                },
                "required": ["user_query", "limit"],
            },
        }
    ]

In [ ]:
messages = [{'role': 'system', 'content': """You are a system which make a JSON query for mongodb using the user input to interpretate it into a JSON, you must generate it.
Always use ignore case and regex, even the user does not say it"""}]
user_message="Who candidates are female?"
messages.append({"role": "user", "content": user_message})
completion= openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    max_tokens=100,
)
response_function_calling = completion.choices[0].message
function_name = (response_function_calling['function_call'])['name']
print(function_name)
parameters = eval(response_function_calling['function_call']['arguments'])
print(parameters)

search_candidates_by
{'user_query': '{"cv_metadata.genre": {"$regex":"female", "$options": "i"}}', 'limit': 3}


In [ ]:
available_functions[function_name](**parameters)

[{'cv_metadata': {'source': '/content/cv-ejemplo-disenador-grafico.pdf',
   'page': 0,
   'full_name': 'Raquel Vázquez García',
   'proffesion_field': 'Graphic Designer',
   'country': 'Peru',
   'genre': 'Female',
   'languages': ['Spanish', 'English'],
   'years_experience': '2',
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': [],
   'hard_skills': ['CorelDRAW', 'Adobe Illustrator', 'Affinity Publisher']}},
 {'cv_metadata': {'source': '/content/cv-ejemplo-educacion-da4453.pdf',
   'page': 0,
   'full_name': 'Samanta Torres',
   'proffesion_field': 'Education',
   'country': 'Lima',
   'genre': 'Female',
   'languages': ['Spanish', 'English', 'German'],
   'years_experience': '4',
   'email': 'ejemplo@cvmaker.pe',
   'soft_skills': ['Patience', 'Persuasion', 'Teamwork'],
   'hard_skills': ['Communication Skills']}},
 {'cv_metadata': {'source': '/content/cv-ejemplo-gerente-sucursal.pdf',
   'page': 0,
   'full_name': 'Ana López',
   'proffesion_field': 'Branch Manager',
   'country

In [ ]:
def ask_to_candidates_bot(user_input):
    messages = [{'role': 'system', 'content': """You are a system which make a JSON query for mongodb using the user input to interpretate it into a JSON, you must generate it.
You must always generate it, applying ignore case and regex for every field, even if the user does not specify them explicitly"""}]
    messages.append({"role": "user", "content": user_input})
    completion = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=messages,
        functions=functions
    )
    response_function_calling = completion.choices[0]['message']
    if response_function_calling is not None and response_function_calling.get('function_call'):
        # If a function call is detected
        function_name = response_function_calling['function_call']['name']
        function_args = json.loads(response_function_calling["function_call"]["arguments"])
        function_response = available_functions[function_name](**function_args)

        print(f'arguments:\n{function_args}\n{"-"*70}\n')
        print(f'query result:\n{function_response}\n{"-"*70}\n')

        if function_response:
            messages.append(response_function_calling)
            messages.append({
                        "role": "function",
                        "name": function_name,
                        "content": str(function_response),
                    })
            bot_response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=messages,
                functions=functions
            )
            return bot_response.choices[0]["message"]["content"].strip()
        else:
          return "No results were found in the search."
    else:
        # If no function call detected, return the initial completion
        return completion.choices[0]["message"]["content"].strip()


In [ ]:
from IPython.display import Markdown
Markdown(ask_to_candidates_bot("Who candidates has less than 5 years of experiences?"))

arguments:
{'user_query': '{"cv_metadata.years_experience": {"$lt": 5}}', 'limit': 3}
----------------------------------------------------------------------

query result:
[{'cv_metadata': {'source': '/content/cv-ejemplo-gerente-sucursal.pdf', 'page': 0, 'full_name': 'Ana López', 'proffesion_field': 'Branch Manager', 'country': 'Lima', 'genre': 'Female', 'languages': [], 'years_experience': 4, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Customer Service', 'Stress Management', 'Leadership'], 'hard_skills': ['Organization', 'Financial Skills', 'Marketing']}}, {'cv_metadata': {'source': '/content/cv-ejemplo-ventas-e9e9e9.pdf', 'page': 0, 'full_name': 'Romina Fernández', 'proffesion_field': 'Sales Executive', 'country': 'Lima', 'genre': 'Female', 'languages': ['Spanish', 'English'], 'years_experience': 3, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Customer Service', 'Interpersonal Skills', 'Communication'], 'hard_skills': ['Sales', 'Inventory Management', 'Shipping Processing']}}]

The candidates who have less than 5 years of experience are:

1. Ana López
   - Profession: Branch Manager
   - Location: Lima
   - Gender: Female
   - Years of Experience: 4
   - Email: ejemplo@cvmaker.pe
   - Soft Skills: Customer Service, Stress Management, Leadership
   - Hard Skills: Organization, Financial Skills, Marketing

2. Romina Fernández
   - Profession: Sales Executive
   - Location: Lima
   - Gender: Female
   - Languages: Spanish, English
   - Years of Experience: 3
   - Email: ejemplo@cvmaker.pe
   - Soft Skills: Customer Service, Interpersonal Skills, Communication
   - Hard Skills: Sales, Inventory Management, Shipping Processing

In [ ]:
from IPython.display import Markdown
Markdown(ask_to_candidates_bot("Who candidates has less than 5 years of experiences and are male?"))

arguments:
{'user_query': '{"cv_metadata.years_experience": {"$lt": 5}, "cv_metadata.genre": {"$regex": "^male$", "$options": "i"}}', 'limit': 3}
----------------------------------------------------------------------

query result:
[]
----------------------------------------------------------------------



No results were found in the search.

In [ ]:
functions = [
        {
            "name": "search_candidates_by",
            "description": "Get a list of candidates which match better with the user's intent",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_query": {
                        "type": "string",
                        "description": """"
A JSON string query for mongodb and always use regex and ignore case as options for each string field,
You must always generate it, applying ignore case and regex for every field, use "$regex", "$options"
The collection structure has a field 'cv_metadata' dictionary which contains:

  - full_name: string
  - proffesion_field: string
  - country: string
  - genre: string
  - languages: list(string)
  - years_experience: integer
  - email: string
  - soft_skills: list(string)
  - hard_skills: list(string)

  for example:
    - {"cv_metadata.years_experience": {"$gt": 10, "$lt": 15}, "cv_metadata.hard_skills": {"$regex":"regulations|law", "$options": "i"} }
    - {"cv_metadata.field": {"$regex":"value", "$options": "i"}}
                        """,
                    },
                    "limit": {
                        "type": "integer",
                        "description": """
The number of candidates the user is looking for is set to 3 by default.
If the user requests multiple candidates.
If the user only requests one candidate or uses the singular form, it is set to 1 by default.
If the user explicitly mentions the number of candidates, that value is returned
"""
                    }
                },
                "required": ["user_query", "limit"],
            },
        }
    ]

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)
model = ChatOpenAI(temperature=0).bind(functions=functions)

model = model.bind(functions=functions)

runnable = prompt | model

result = runnable.invoke({"input": "Who candidates has less than 5 years of experiences, has law knowledge and are male?"})
result

AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_candidates_by', 'arguments': '{\n  "user_query": "{\\"cv_metadata.years_experience\\": {\\"$lt\\": 5}, \\"cv_metadata.hard_skills\\": {\\"$regex\\":\\"law\\", \\"$options\\": \\"i\\"}, \\"cv_metadata.genre\\": {\\"$regex\\":\\"male\\", \\"$options\\": \\"i\\"}}",\n  "limit": 3\n}'}})

In [ ]:
result.to_json()['kwargs']['additional_kwargs']

{'function_call': {'name': 'search_candidates_by',
  'arguments': '{\n  "user_query": "{\\"cv_metadata.years_experience\\": {\\"$lt\\": 5}, \\"cv_metadata.hard_skills\\": {\\"$regex\\":\\"law\\", \\"$options\\": \\"i\\"}, \\"cv_metadata.genre\\": {\\"$regex\\":\\"male\\", \\"$options\\": \\"i\\"}}",\n  "limit": 3\n}'}}

In [ ]:
result.additional_kwargs['function_call']['arguments']

'{\n  "user_query": "{\\"cv_metadata.years_experience\\": {\\"$lt\\": 5}, \\"cv_metadata.hard_skills\\": {\\"$regex\\":\\"law\\", \\"$options\\": \\"i\\"}, \\"cv_metadata.genre\\": {\\"$regex\\":\\"male\\", \\"$options\\": \\"i\\"}}",\n  "limit": 3\n}'

In [ ]:
json.loads(result.additional_kwargs['function_call']['arguments'])

{'user_query': '{"cv_metadata.years_experience": {"$lt": 5}, "cv_metadata.hard_skills": {"$regex":"law", "$options": "i"}, "cv_metadata.genre": {"$regex":"male", "$options": "i"}}',
 'limit': 3}

In [ ]:
runnable.invoke({"input": "hello"})

AIMessage(content='Hello! How can I assist you today?')

In [ ]:
result = runnable.invoke({"input": "Who candidates has less than 5 years of experiences or has law knowledge or is male?"})
json.loads(result.additional_kwargs['function_call']['arguments'])

{'user_query': '{"$or": [{"cv_metadata.years_experience": {"$lt": 5}}, {"cv_metadata.hard_skills": {"$regex": "law", "$options": "i"}}, {"cv_metadata.genre": "male"}]}',
 'limit': 3}

In [ ]:
from langchain.llms import OpenAI
import json

chain = model | StrOutputParser() | json.loads

chain.invoke("Who candidates has less than 5 years of experiences?, return in json format")

JSONDecodeError: ignored